In [0]:
%pip install xgboost==3.4.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 73.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.4/206.4 MB 163.5 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [xgboost]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%pip install xgboost mlflow

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 153.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 122.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 99.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 100.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 105.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 826.6/826.6 kB 41.8 MB/s  0:00:00
  Attempting uninstall: wcwidth
    Found existing installation: wcwidth 0.2.13
    Not uninstalling wcwidth at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-1d6aa7b2-1b26-47ae-8e32-64ac9ff698e7
    Can't uninstall 'wcwidth'. No files were found to uninstall.
  Attempting uninstall: blinker
    Found existing installation: blinker 1.7.0
    Not uninstalling blinker at /usr/lib/python3/dist-packages, outside environment /local_disk0/.ephemeral_n

In [0]:
import mlflow
import mlflow.xgboost
from mlflow.tracking import MlflowClient

# 1. Apontar para o Unity Catalog e buscar a versão mais recente do modelo
mlflow.set_registry_uri("databricks-uc")
model_name = "workspace.default.fraude_cartao_xgboost"

client = MlflowClient()
versoes = client.search_model_versions(f"name='{model_name}'")
ultima_versao = max(int(v.version) for v in versoes)

model_uri = f"models:/{model_name}/{ultima_versao}"
print(f"Carregando modelo: {model_uri}")
model = mlflow.xgboost.load_model(model_uri)

# 2. Ler a tabela de features gerada na Fase 3
df_pd = spark.read.table("workspace.default.transacoes_features").toPandas()
X = df_pd.drop("FRAUD", axis=1).astype("float64")

# 3. Gerar predições e probabilidades em lote
df_pd["PREDICAO_FRAUDE"] = model.predict(X).astype(int)
df_pd["PROBABILIDADE_FRAUDE"] = model.predict_proba(X)[:, 1].round(4)

# 4. Salvar resultado em tabela Delta no Unity Catalog
df_pred = spark.createDataFrame(df_pd)
df_pred.write.mode("overwrite").saveAsTable("workspace.default.transacoes_predicoes_batch")

print("Inferência batch concluída e salva em workspace.default.transacoes_predicoes_batch")
df_pred.select("TX_AMOUNT", "RISCO_TERMINAL", "FRAUD", "PREDICAO_FRAUDE", "PROBABILIDADE_FRAUDE").show(10)

Carregando modelo: models:/workspace.default.fraude_cartao_xgboost/1


Inferência batch concluída e salva em workspace.default.transacoes_predicoes_batch
+---------+--------------+-----+---------------+--------------------+
|TX_AMOUNT|RISCO_TERMINAL|FRAUD|PREDICAO_FRAUDE|PROBABILIDADE_FRAUDE|
+---------+--------------+-----+---------------+--------------------+
|    30.06|           0.0|    0|              0|              0.0042|
|    22.67|           0.0|    0|              0|              0.0042|
|   151.25|           0.0|    0|              0|              0.0042|
|    23.96|           0.0|    0|              0|              0.0042|
|    150.1|           0.0|    0|              0|              0.0042|
|    200.1|           0.0|    0|              0|              0.0042|
|   263.18|           0.0|    0|              0|              0.0042|
|   173.35|           0.0|    0|              0|              0.0042|
|   158.58|           0.0|    0|              0|              0.0042|
|    36.13|           0.0|    0|              0|              0.0042|
+------

In [0]:
import mlflow
from pyspark.sql import functions as F

mlflow.set_registry_uri("databricks-uc")

# 1. Carrega o modelo em produção diretamente pelo alias do Unity Catalog
model_uri = "models:/workspace.default.fraude_cartao_xgboost@champion"
model = mlflow.xgboost.load_model(model_uri)

# 2. Lê a tabela de features (camada Silver)
df_spark = spark.read.table("workspace.default.transacoes_features")
pdf = df_spark.toPandas()

feature_cols = [
    "TX_AMOUNT", "TX_TIME_HOURS", "TX_DAY_OF_WEEK",
    "TX_COUNT_1H", "TX_SUM_1H", "TX_COUNT_24H", "TX_SUM_24H",
    "RISCO_TERMINAL"
]

# 3. Gera classe prevista e probabilidade (alinhado ao schema float64 do treino)
X = pdf[feature_cols].astype("float64")
pdf["PROBABILIDADE_FRAUDE"] = model.predict_proba(X)[:, 1]
pdf["PREDICAO_FRAUDE"] = (pdf["PROBABILIDADE_FRAUDE"] >= 0.50).astype(int)

# 4. Grava o resultado governado como tabela Delta (camada Gold)
df_gold = (
    spark.createDataFrame(pdf)
         .withColumn("DT_INFERENCIA", F.current_timestamp())
)

df_gold.write.mode("overwrite").saveAsTable("workspace.default.transacoes_predicoes")
display(spark.read.table("workspace.default.transacoes_predicoes").limit(10))

TX_AMOUNT,TX_TIME_HOURS,TX_DAY_OF_WEEK,TX_COUNT_1H,TX_SUM_1H,TX_COUNT_24H,TX_SUM_24H,RISCO_TERMINAL,FRAUD,PROBABILIDADE_FRAUDE,PREDICAO_FRAUDE,DT_INFERENCIA
30.06,9,6,0,0.0,0,0.0,0.0,0,0.0042202566,0,2026-11-23T22:55:29.968Z
22.67,20,6,0,0.0,0,0.0,0.0,0,0.0042202566,0,2026-11-23T22:55:29.968Z
151.25,8,7,0,0.0,3,360.12,0.0,0,0.0042202566,0,2026-11-23T22:55:29.968Z
23.96,9,7,0,0.0,1,22.67,0.0,0,0.0042202566,0,2026-11-23T22:55:29.968Z
150.1,11,7,0,0.0,4,511.37,0.0,0,0.0042202566,0,2026-11-23T22:55:29.968Z
200.1,16,7,1,143.57,2,275.21,0.0,0,0.0042202566,0,2026-11-23T22:55:29.968Z
263.18,17,7,0,0.0,2,476.87,0.0,0,0.0042202566,0,2026-11-23T22:55:29.968Z
173.35,21,7,0,0.0,0,0.0,0.0,0,0.0042202566,0,2026-11-23T22:55:29.968Z
158.58,8,6,0,0.0,0,0.0,0.0,0,0.0042202566,0,2026-11-23T22:55:29.968Z
36.13,13,6,0,0.0,1,39.82,0.0,0,0.0042202566,0,2026-11-23T22:55:29.968Z
